# Short-Rate Calibration Lab

Module: Term Structure and Interest Rate Models

## Lesson summary

The Vasicek model can be calibrated through an AR(1) bridge, while the CIR model requires extra care because volatility depends on the rate level. This lab estimates Vasicek parameters from a deterministic short-rate series, simulates exact Vasicek paths, checks the CIR Feller condition, and compares it with Full Truncation Euler-Maruyama simulation.

## Learning objectives

By the end of this lesson, students should be able to:

- estimate an AR(1) representation of the short rate;
- translate AR(1) parameters into Vasicek parameters;
- simulate exact Vasicek paths from calibrated parameters;
- evaluate the CIR Feller condition;
- simulate CIR paths with a boundary-safe discretization;
- identify model risk in one-factor short-rate models.

## Python setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.term_structure import (
    feller_condition,
    simulate_cir_full_truncation,
    simulate_vasicek_exact,
    synthetic_mexican_yield_curve_history,
    vasicek_ols_calibration,
)

## Short-rate proxy

Use the three-month point from the synthetic curve as a classroom proxy for the short rate.

In [ ]:
curve_history = synthetic_mexican_yield_curve_history()
short_rate = curve_history["0.25Y"]
short_rate.tail()

In [ ]:
short_rate.plot(figsize=(10, 4), title="Synthetic Short-Rate Proxy")
plt.xlabel("Date")
plt.ylabel("Rate")
plt.grid(True, alpha=0.3)
plt.show()

## Vasicek calibration through AR(1)

The discrete bridge is:

$$
r_{t+\Delta t} = c + \beta r_t + \epsilon_t.
$$

The continuous-time parameters are:

$$
\kappa = -\frac{\ln(\beta)}{\Delta t},
\qquad
\theta = \frac{c}{1-\beta},
\qquad
\sigma =
\sqrt{\frac{2\kappa\sigma_\epsilon^2}{1-\beta^2}}.
$$

In [ ]:
calibration = vasicek_ols_calibration(short_rate, dt=1 / 252)
pd.Series(calibration)

## Vasicek path simulation

In [ ]:
vasicek_paths = simulate_vasicek_exact(
    r0=float(short_rate.iloc[-1]),
    kappa=calibration["kappa"],
    theta=calibration["theta"],
    sigma=calibration["sigma"],
    years=3,
    paths=200,
)

vasicek_paths.iloc[:, :20].plot(figsize=(10, 4), legend=False, alpha=0.35, title="Calibrated Vasicek Paths")
plt.xlabel("Years")
plt.ylabel("Rate")
plt.grid(True, alpha=0.3)
plt.show()

## CIR Feller condition

The CIR model is:

$$
dr_t = \kappa(\theta-r_t)dt + \sigma\sqrt{r_t}dW_t.
$$

The Feller condition is:

$$
2\kappa\theta \geq \sigma^2.
$$

In [ ]:
cir_params = {
    "r0": float(short_rate.iloc[-1]),
    "kappa": 0.75,
    "theta": 0.075,
    "sigma": 0.10,
}

pd.Series(
    {
        **cir_params,
        "feller_condition_satisfied": feller_condition(
            cir_params["kappa"],
            cir_params["theta"],
            cir_params["sigma"],
        ),
    }
)

## CIR Full Truncation Euler-Maruyama

Full truncation applies the positive part of the previous rate inside the square-root term:

$$
r_t^+ = \max(r_t, 0).
$$

In [ ]:
cir_paths = simulate_cir_full_truncation(
    r0=cir_params["r0"],
    kappa=cir_params["kappa"],
    theta=cir_params["theta"],
    sigma=cir_params["sigma"],
    years=3,
    paths=200,
)

cir_paths.iloc[:, :20].plot(figsize=(10, 4), legend=False, alpha=0.35, title="CIR Full Truncation Paths")
plt.xlabel("Years")
plt.ylabel("Rate")
plt.grid(True, alpha=0.3)
plt.show()

## Distribution comparison

In [ ]:
summary = pd.DataFrame(
    {
        "vasicek_final": vasicek_paths.iloc[-1],
        "cir_final": cir_paths.iloc[-1],
    }
).describe(percentiles=[0.05, 0.50, 0.95])

summary

## Model risk checklist

| Risk | Diagnostic question |
| --- | --- |
| Parameter uncertainty | Are estimates stable across samples? |
| Calibration error | Does the model match the current curve? |
| Recalibration risk | Do parameters jump sharply when new data arrive? |
| Misspecification risk | Does the model allow behavior that is economically implausible for the use case? |

## Model limitations

- AR(1)-based calibration is a classroom bridge and can be biased by discretization and measurement noise.
- The Feller condition is a parameter restriction, not a complete validation of CIR fit.
- Simulated rate paths should be interpreted with model-risk notes before being used for valuation.